# 06 — Integration & Evaluation

This notebook ties all three modules together and produces the final evaluation:

1. Load trained ResNet-50 and run inference on test set
2. Full evaluation: QWK, AUC-ROC, F1, confusion matrix
3. Baseline comparison (ResNet-50 vs VGG-16 vs Scratch)
4. Attach CNN predictions to patient network
5. Generate final network visualisation with CNN-predicted DR grades
6. Produce results summary table

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.utils.config import load_config, get_device
from src.models import build_resnet50
from src.data import get_val_transforms
from src.data.dataset import create_data_loaders
from src.evaluation import compute_all_metrics, plot_confusion_matrix, plot_roc_curves

cfg    = load_config('../configs/default.yaml')
device = get_device()
print(f'Device: {device}')

## 1. Load Best Model & Inference

In [ ]:
model = build_resnet50(num_classes=5, fc_hidden=512, dropout=0.3,
                        pretrained=False).to(device)
ckpt = torch.load('../outputs/models/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]} | Val QWK: {ckpt["val_qwk"]:.4f}')

In [ ]:
_, _, test_loader = create_data_loaders(
    train_csv=cfg['paths']['train_csv'],
    val_csv=cfg['paths']['val_csv'],
    test_csv=cfg['paths']['test_csv'],
    train_image_dir=cfg['paths']['train_images'],
    val_image_dir=cfg['paths']['val_images'],
    test_image_dir=cfg['paths']['test_images'],
    train_transform=get_val_transforms(224),
    val_transform=get_val_transforms(224),
    image_size=224, batch_size=32,
)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
print(f'Inference complete on {len(all_labels)} test images')

## 2. Full Evaluation

In [ ]:
metrics = compute_all_metrics(all_labels, all_preds, all_probs)

print('=' * 50)
print('  ResNet-50 Test Set Results')
print('=' * 50)
for k, v in metrics.items():
    if v is not None:
        print(f'  {k:30s}: {v:.4f}')

In [ ]:
plot_confusion_matrix(all_labels, all_preds,
                       '../outputs/figures/confusion_matrix.png')
plot_roc_curves(all_labels, all_probs,
                 '../outputs/figures/roc_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
import cv2
for ax, fname, title in zip(axes,
    ['../outputs/figures/confusion_matrix.png',
     '../outputs/figures/roc_curves.png'],
    ['Confusion Matrix', 'ROC Curves']):
    img = cv2.cvtColor(cv2.imread(fname), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Baseline Comparison

In [ ]:
# Placeholder — fill in after training VGG-16 and scratch model in Notebook 03
results = {
    'ResNet-50 (ours)': metrics,
    # 'VGG-16': metrics_vgg,
    # 'Scratch':  metrics_scratch,
}

summary_rows = []
for model_name, m in results.items():
    summary_rows.append({
        'Model': model_name,
        'QWK ↑': round(m.get('quadratic_weighted_kappa', 0), 4),
        'AUC-ROC ↑': round(m.get('auroc', 0), 4),
        'F1 Weighted ↑': round(m.get('f1_weighted', 0), 4),
        'Accuracy ↑': round(m.get('accuracy', 0), 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../outputs/results/baseline_comparison.csv', index=False)
print(summary_df.to_string(index=False))

## 4. Integrate CNN Predictions into Patient Network

In [ ]:
from src.network.analysis import NetworkAnalyzer
from src.utils.config import load_config

clinical_df = pd.read_csv('../data/metadata/clinical_metadata.csv')
feature_cols = [c for c in cfg['network']['clinical_features']
                if c in clinical_df.columns]

# For patients in metadata, use CNN-predicted grades
# (here we use the stored dr_grade as a proxy; in production,
#  map patient_id → CNN prediction)
dr_grades = clinical_df['dr_grade'].values if 'dr_grade' in clinical_df.columns else None

analyzer = NetworkAnalyzer(
    clinical_df=clinical_df,
    feature_columns=feature_cols,
    threshold=cfg['network']['edge_threshold'],
    dr_grades=dr_grades,
)
summary = analyzer.summary()
print('Network Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 5. Final Results Summary

In [ ]:
print('\n' + '=' * 60)
print('  Praxis — Final Results Summary')
print('=' * 60)

print('\n  CNN Performance (Test Set):')
for k, v in metrics.items():
    if v is not None:
        print(f'    {k}: {v:.4f}')

print('\n  Patient Network:')
for k, v in summary.items():
    if v is not None:
        val_str = f'{v:.4f}' if isinstance(v, float) else str(v)
        print(f'    {k}: {val_str}')

print('\n  Outputs saved to outputs/ directory')
print('  Run the Streamlit dashboard:  streamlit run app/streamlit_app.py')